# Finans / Borsa Zaman Serisi Analizi (Market Predictor)
Bu çalışmada, Yahoo Finance (`yfinance`) üzerinden gerçek borsa verilerini (Örn: Apple - AAPL) çekip, geçmiş kapanış fiyatlarına dayanarak gelecek trendleri tahmin edeceğiz.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

plt.style.use('dark_background')

## 1. Veri Çekme (Data Acquisition)
`yfinance` kütüphanesini kullanarak Apple'ın (AAPL) son 5 yıllık hisse senedi verilerini indirelim.

In [ ]:
# Son 5 yıllık veriyi indir
ticker = 'AAPL'
data = yf.download(ticker, start='2019-01-01', end='2024-01-01')

# Sadece ihtiyacımız olan 'Close' (Kapanış) fiyatlarını alalım
df = data[['Close']].copy()
df.dropna(inplace=True)
display(df.head())


## 2. Keşifsel Veri Analizi ve Görselleştirme (EDA)
Son 5 yılda fiyat nasıl değişmiş?

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(df.index, df['Close'], color='cyan', label='Kapanış Fiyatı')
plt.title(f'{ticker} Hisse Senedi Fiyat Grafiği')
plt.xlabel('Tarih')
plt.ylabel('Fiyat (USD)')
plt.legend()
plt.show()

## 3. Özellik Mühendisliği (Feature Engineering)
Basit bir Makine Öğrenmesi (Linear Regression) modeli için geçmiş verilerden "Gelecekteki N. günü" hedef (target) olarak belirleyelim.

In [ ]:
# Kaç gün sonrasını tahmin edeceğiz? (Örn: 30 gün)
forecast_out = 30

# Hedef sütunu (target): Kapanış fiyatını 'forecast_out' kadar yukarı kaydırıyoruz
df['Prediction'] = df[['Close']].shift(-forecast_out)

# X (Özellikler) matrisini oluştur
X = np.array(df.drop(['Prediction'], axis=1))
X = X[:-forecast_out] # Son 30 günü çıkar çünkü hedef(label) yok

# y (Hedef) vektörünü oluştur
y = np.array(df['Prediction'])
y = y[:-forecast_out]

# Veriyi Eğitim (Train) ve Test olarak böl
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Eğitim verisi boyutu: {X_train.shape}")
print(f"Test verisi boyutu: {X_test.shape}")

## 4. Model Eğitimi (Linear Regression)

In [ ]:
# Modeli eğit
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Test seti üzerinde doğruluk puanı (R^2 Score)
score = lr_model.score(X_test, y_test)
print(f"Model R^2 Puanı: {score:.4f}")

## 5. Tahmin Yapma ve Görselleştirme
En son 30 günün (henüz cevabını bilmediğimiz) tahminlerini yapalım ve gerçek verilerle görselleştirelim.

In [ ]:
# Tahmin edilecek son 30 günlük X verisi
X_forecast = np.array(df.drop(['Prediction'], axis=1))[-forecast_out:]
lr_prediction = lr_model.predict(X_forecast)

# Tahmin sonuçlarını DataFrame'e ekle
df['Forecast_LR'] = np.nan
last_date = df.iloc[-1].name
last_unix = last_date.timestamp()
one_day = 86400
next_unix = last_unix + one_day

# Gelecek 30 gün için indeks oluştur
for i in lr_prediction:
    next_date = pd.to_datetime(next_unix, unit='s')
    df.loc[next_date] = [np.nan, np.nan, i]
    next_unix += one_day

plt.figure(figsize=(14, 6))
plt.plot(df['Close'], color='cyan', label='Gerçek Fiyat')
plt.plot(df['Forecast_LR'], color='magenta', label='Tahmin Edilen (Gelecek 30 Gün)')
plt.xlim(df.index[-100], df.index[-1]) # Sadece son kısımları göster
plt.title(f'{ticker} Hisse Senedi Fiyat Tahmini')
plt.xlabel('Tarih')
plt.ylabel('Fiyat (USD)')
plt.legend()
plt.show()

## 6. Özet ve Sonuçlar
### Q&A
- **Soru:** Sadece geçmiş kapanış fiyatına bakarak borsayı kesin olarak tahmin edebilir miyiz?
- **Cevap:** Hayır, finansal piyasalar stokastiktir. Lineer Regresyon sadece mevcut trendi doğrusal olarak uzatır, ani piyasa şoklarını veya haberleri öngöremez.

### Data Analysis Key Findings
- Apple hissesi (AAPL) son 5 yılda net bir pozitif doğrusal trend sergilediği için Lineer Regresyon modeli genel yükseliş yönünü yakalayabilmektedir.
- R^2 skoru, hissenin kendi geçmiş verisiyle doğrusal bir ilişkisi olduğunu onaylamaktadır.

### Insights or Next Steps
- Modele finansal haberlerin duygu analizleri (Sentiment Analysis) eklenerek (hibrit model) başarı artırılabilir.
- Zaman serileri için daha gelişmiş modeller olan ARIMA, Prophet veya LSTM (Yapay Sinir Ağları) denenmelidir.